# tcOCR — bản test cho Colab **Free GPU (Tesla T4)**

Bản này tối ưu cho **free tier**: Paddle chạy CPU (chắc ăn), VietOCR + Qwen2.5-VL-3B + ProtonX chạy GPU T4.

### Free tier có gì / lưu ý
- GPU thường là **Tesla T4 16GB**, **không đảm bảo** luôn có (tùy hạn ngạch Google cấp lúc đó).
- Tự ngắt khi **idle ~90 phút**, tối đa **~12h/phiên**. Xài nặng có thể bị hạ về CPU.
- Hợp để **test**, KHÔNG hợp chạy production 60k trang/tháng.

### Bật GPU
**Runtime → Change runtime type → Hardware accelerator = GPU (T4) → Save.**

⚠️ Link Gradio `share=True` công khai — chỉ test **tài liệu tài chính công khai**, KHÔNG dùng data thật của khách.

## 1. Kiểm tra đã có GPU chưa

In [ ]:
!nvidia-smi -L || echo 'CHUA CO GPU -> Runtime -> Change runtime type -> GPU (T4)'

## 2. Lấy code

In [ ]:
import os
if not os.path.exists('tcocr/tcocr'):
    !git clone -b claude/ocr-finance-99-percent-b5o5rh https://github.com/vnkiddev/tcocr.git
%cd tcocr
!git pull

## 3. Cài thư viện (~6-9 phút lần đầu)

Paddle bản **CPU** (bản `-gpu` trên PyPI hay lỗi build). torch giữ nguyên bản GPU có sẵn của Colab.

In [ ]:
!pip install -q numpy opencv-python-headless pillow PyMuPDF pdfplumber gradio
!pip uninstall -y -q paddlepaddle-gpu paddlepaddle 2>/dev/null
!pip install -q paddlepaddle paddleocr
!pip install -q vietocr
!pip install -q 'transformers>=4.49' accelerate qwen-vl-utils bitsandbytes sentencepiece
print('Cai xong.')
print('>>> BAT BUOC: Runtime -> Restart session MOT LAN, roi chay tiep tu cell 4. <<<')

## 4. Kiểm tra import + GPU (chạy SAU khi Restart session)

In [ ]:
%cd /content/tcocr
import torch, paddle, paddleocr
print('torch cuda :', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('paddle     :', paddle.__version__)
print('paddleocr  :', paddleocr.__version__)
from tcocr.pipeline import OCRPipeline
print('tcocr OK')

## 5A. Gradio UI — nhẹ (escalation TẮT)

Chạy nhanh, ít tốn VRAM. So Paddle vs VietOCR, bật/tắt correction. Muốn thử VLM thì dùng cell 5B.

In [ ]:
from app import build_ui
build_ui().launch(share=True)

## 5B. (Tuỳ chọn) Test escalation Vision-LLM **Qwen2.5-VL-3B 4-bit** trên T4

3B 4-bit ~4GB VRAM, vừa T4. Lần đầu tải model ~4GB (vài phút). Chạy trực tiếp trên 1 file, không qua UI cho gọn VRAM.

> Nếu OOM: đổi sang xử lý từng trang, hoặc giảm `pdf_dpi`. 7B chỉ nên thử khi T4 còn trống nhiều.

In [ ]:
from tcocr.config import PipelineConfig
from tcocr.pipeline import OCRPipeline

cfg = PipelineConfig(
    ocr_backend='paddle',
    escalation_backend='local_vlm',
    escalation_kwargs={'model_id': 'Qwen/Qwen2.5-VL-3B-Instruct', 'load_in_4bit': True},
    escalate_confidence_threshold=0.85,   # đẩy nhiều vùng lên VLM để thấy khác biệt
    correction_backend='protonx',
    pdf_dpi=150,                           # giảm DPI cho nhẹ VRAM trên free tier
)
pipe = OCRPipeline(cfg)
doc = pipe.process_pdf('scan.pdf')          # đổi đường dẫn file scan của bạn
for p in doc.pages:
    print(f'--- Trang {p.page_index} | flags: {len(p.validation_flags)} ---')
    print(p.full_text()[:1500])

## 6. Benchmark field-level (Paddle vs VietOCR)

Cần **PDF scan** (đầu vào) + **PDF gốc digital** (nguồn gold).

In [ ]:
from tcocr.config import PipelineConfig
from tcocr.pipeline import OCRPipeline
from tcocr.benchmark.runner import run_pair

for backend in ['paddle', 'vietocr']:
    cfg = PipelineConfig(ocr_backend=backend, escalation_backend='null', correction_backend='null')
    rep = run_pair(OCRPipeline(cfg), 'scan.pdf', 'goc.pdf')   # đổi đường dẫn 2 file
    print(f'\n===== {backend} =====')
    print(rep.aggregate())

### Dọn VRAM khi cần (giữa các lần đổi model)

In [ ]:
import gc, torch
gc.collect(); torch.cuda.empty_cache()
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv